# COGS515 Lab: Heuristics for the 8-Puzzle

**Week 3**

> Check Python Bootcamp first if you are not proficient with Python.

**How this lab works**
- ✏️ means answer on the worksheet (pen and paper). 💻 means run a cell.
- **Predict first, then run.** Write your prediction on the worksheet *before* the cell is run.




## 0. Setup 💻 (1 min)

Run the next cell. It is last week's code: the `EightPuzzle` problem, `Node`, `expand`, `report` and `show`, and puzzles A, B and C. Nothing in it is new.

In [ ]:
from collections import deque
import heapq
from itertools import count


class EightPuzzle:
    # Positions the blank can move to, for each position of the blank
    MOVES = ((1, 3), (0, 2, 4), (1, 5),
             (0, 4, 6), (1, 3, 5, 7), (2, 4, 8),
             (3, 7), (4, 6, 8), (5, 7))

    def __init__(self, initial, goal=(0, 1, 2, 3, 4, 5, 6, 7, 8)):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        return self.MOVES[state.index(0)]

    def result(self, state, action):
        s = list(state)
        blank = state.index(0)
        s[blank], s[action] = s[action], 0
        return tuple(s)

    def goal_test(self, state):
        return state == self.goal

    def action_cost(self, state1, action, state2):
        return 1


def show(*states):
    """Print one or more states side by side, with _ for the blank."""
    for row in range(3):
        print("   ".join(" ".join(str(t) if t else "_" for t in s[3 * row: 3 * row + 3]) for s in states))
    print()


class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = parent.depth + 1 if parent else 0


def expand(problem, node):
    s = node.state
    for action in problem.actions(s):
        child_state = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, child_state)
        yield Node(child_state, node, action, cost)


def report(node, expanded, max_frontier):
    if node is None:
        print(f"No solution | nodes expanded: {expanded:,}")
    else:
        print(f"Solution: {node.depth:,} moves | nodes expanded: {expanded:,} | largest frontier: {max_frontier:,}")


A = (1, 2, 5, 3, 4, 8, 0, 6, 7)    # 6 moves
B = (5, 6, 2, 1, 4, 3, 0, 7, 8)    # 20 moves
C = (7, 2, 4, 5, 0, 6, 8, 3, 1)    # 26 moves
show(A, B, C)

## 1. Heuristics by hand ✏️ (6 min)

A **heuristic** $h(n)$ estimates the cost of the cheapest path from the state of node $n$ to a goal. Two classic heuristics for the 8-puzzle:

- **Misplaced tiles**, $h_1$: the number of tiles that are not on their goal square. The blank does not count.
- **Manhattan distance**, $h_2$: for each tile, count the rows plus the columns between its square and its goal square, then add these up. The blank does not count.

```
  Puzzle A        Puzzle C        Goal
   1 2 5           7 2 4          _ 1 2
   3 4 8           5 _ 6          3 4 5
   _ 6 7           8 3 1          6 7 8
```

- ✏️ **Task 1a.** Compute $h_1$ and $h_2$ for puzzle A. Last week BFS showed that A needs 6 moves. Does either heuristic overestimate?
- ✏️ **Task 1b.** Compute $h_1$ and $h_2$ for puzzle C, which needs 26 moves. *Tip:* for $h_2$, fill in the tile-by-tile row on the worksheet.
- ✏️ **Task 1c.** Is $h_2 \ge h_1$ for every state? Why?

Then run the two cells below to check 1a and 1b.

In [ ]:
# In our goal, tile t belongs on square t. Square s is in row s // 3 and column s % 3.

def h_misplaced(state):
    """h1: number of misplaced tiles (the blank does not count)."""
    total = 0
    for square, tile in enumerate(state):
        if tile != 0 and tile != square:
            total += 1                                          # each misplaced tile counts 1
    return total


def h_manhattan(state):
    """h2: total Manhattan distance of the tiles (the blank does not count)."""
    total = 0
    for square, tile in enumerate(state):
        if tile != 0:
            total += abs(square // 3 - tile // 3) + abs(square % 3 - tile % 3)    # rows + columns to go
    return total

In [ ]:
print("A: h1 =", h_misplaced(A), "  h2 =", h_manhattan(A))    # Task 1a
print("C: h1 =", h_misplaced(C), "  h2 =", h_manhattan(C))    # Task 1b

## 2. One algorithm, three strategies (9 min)

Uniform-cost search, greedy best-first search and A\* are all **best-first search**: always expand the frontier node with the lowest $f(n)$. Only $f$ changes.

| Strategy | $f(n)$ |
|---|---|
| Uniform-cost search | $g(n)$, the cost of the path so far |
| Greedy best-first search | $h(n)$, the estimated cost still to go |
| A\* | $g(n) + h(n)$ |

`best_first` is last week's `bfs` with three changes:
- the frontier is a **priority queue** (`heapq`) ordered by $f$; when two nodes have the same $f$, the one added first comes out first;
- `reached` maps each state to the **cheapest node** found so far for it, and a child is added if its state is new *or* reached more cheaply (the rule on the lecture's UCS slide). An older, more expensive copy of a state is skipped when it comes out of the queue;
- as last week, the goal test happens when a node is **removed** from the frontier.

In [ ]:
def best_first(problem, f):
    """Expand the frontier node with the lowest f(node) first."""
    node = Node(problem.initial)
    tie = count()                                   # 0, 1, 2, ...: on equal f, the older node comes out first
    frontier = [(f(node), next(tie), node)]         # priority queue (heap) of (f, tie, node)
    reached = {problem.initial: node}               # state -> cheapest node found so far
    expanded, max_frontier = 0, 1
    while frontier:
        _, _, node = heapq.heappop(frontier)        # the node with the lowest f
        if node.path_cost > reached[node.state].path_cost:
            continue                                # a cheaper path to this state was found later: skip
        expanded += 1
        if problem.goal_test(node.state):
            report(node, expanded, max_frontier)
            return node
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                heapq.heappush(frontier, (f(child), next(tie), child))
        max_frontier = max(max_frontier, len(frontier))
    report(None, expanded, max_frontier)
    return None


def ucs(problem):
    return best_first(problem, lambda n: n.path_cost)                 # f = g


def greedy(problem, h):
    return best_first(problem, lambda n: h(n.state))                  # f = h


def astar(problem, h):
    return best_first(problem, lambda n: n.path_cost + h(n.state))    # f = g + h

- ✏️ **Task 2a.** Before running each of the three cells below, predict for puzzle C: will the solution be the shortest (26 moves)? Will the search expand more or fewer nodes than last week's BFS (169,741), and roughly how many? Write your predictions in the table on the worksheet, then fill in the results.

In [ ]:
ucs(EightPuzzle(C));

In [ ]:
greedy(EightPuzzle(C), h_manhattan);

In [ ]:
astar(EightPuzzle(C), h_manhattan);

- ✏️ **Task 2b.** UCS expanded exactly as many nodes as BFS did last week. Why?
- ✏️ **Task 2c.** Greedy search was by far the fastest, but its solution has 46 moves. Why is greedy search not optimal?
- ✏️ **Task 2d.** What would A\* do with $h(n) = 0$ for every node?

## 3. Better heuristics (8 min)

Neither $h_1$ nor $h_2$ ever overestimates, so A\* finds a shortest solution with either one. They differ in how much work that takes.

- ✏️ **Task 3a.** Predict: how many nodes will A\* expand on puzzle C with misplaced tiles ($h_1$)? Compare with Manhattan's 3,978. Then run.

In [ ]:
astar(EightPuzzle(C), h_misplaced);

- ✏️ **Task 3b.** Both runs found 26 moves. Why did $h_2$ need so many fewer nodes? *Hint:* Task 1c.

**Is a heuristic admissible?** The next cell finds the **true cost** (the length of a shortest solution) of every state that can reach the goal. It is last week's breadth-first search, started at the goal and run until every state is reached: every move can be undone, so the distance from the goal to a state is also the distance from that state to the goal. `check_admissible(h)` then compares $h$ with the true cost on all of these states.

In [ ]:
def true_costs(goal=(0, 1, 2, 3, 4, 5, 6, 7, 8)):
    """Breadth-first search from the goal: state -> number of moves to the goal."""
    puzzle = EightPuzzle(goal)
    cost = {goal: 0}
    frontier = deque([goal])
    while frontier:
        state = frontier.popleft()
        for action in puzzle.actions(state):
            child = puzzle.result(state, action)
            if child not in cost:
                cost[child] = cost[state] + 1
                frontier.append(child)
    return cost


TRUE_COST = true_costs()
print(f"{len(TRUE_COST):,} states; the hardest ones need {max(TRUE_COST.values())} moves.")


def check_admissible(h):
    too_high = [s for s in TRUE_COST if h(s) > TRUE_COST[s]]
    if not too_high:
        print("Admissible: never overestimates.")
    else:
        s = too_high[0]                             # the first one found
        print(f"Not admissible: overestimates on {len(too_high):,} of {len(TRUE_COST):,} states.")
        print(f"For example, on this state h = {h(s)}, but the true cost is {TRUE_COST[s]}:")
        show(s)


check_admissible(h_misplaced)
check_admissible(h_manhattan)

- ✏️💻 **Task 3c.** Design a heuristic of your own.
  - **With a laptop:** write `h_mine(state)` in the cell below; your rule of thumb from last week's lab is a good start. The cell after it checks whether it is admissible and runs A\* on puzzle C.
  - **Without a laptop:** for each heuristic in the menu, circle on the worksheet whether you think it is admissible, and predict whether A\* will expand more or fewer nodes than with Manhattan (3,978). The instructor will run the menu cell.

| | Menu heuristic |
|---|---|
| (a) | tiles in the wrong row + tiles in the wrong column |
| (b) | rows + columns from the blank to its goal square (top left) |
| (c) | 2 × misplaced tiles |

In [ ]:
def h_mine(state):
    # TASK 3c: return your estimate of the number of moves from state to the goal
    return None

In [ ]:
if h_mine(C) is None:
    print("Write h_mine first (Task 3c), or try the menu below.")
else:
    check_admissible(h_mine)
    astar(EightPuzzle(C), h_mine)

In [ ]:
def h_rowcol(state):                                # (a)
    total = 0
    for square, tile in enumerate(state):
        if tile != 0:
            if square // 3 != tile // 3:            # wrong row
                total += 1
            if square % 3 != tile % 3:              # wrong column
                total += 1
    return total


def h_blank(state):                                 # (b)
    blank = state.index(0)
    return blank // 3 + blank % 3                   # rows + columns from the blank to square 0


def h_double(state):                                # (c)
    return 2 * h_misplaced(state)


for name, h in [("(a) wrong row + wrong column", h_rowcol),
                ("(b) blank to its goal square", h_blank),
                ("(c) 2 x misplaced tiles", h_double)]:
    print(name)
    check_admissible(h)
    astar(EightPuzzle(C), h)
    print()

## 4. Overestimating on purpose (6 min)

What if we **double** the Manhattan distance? The next cell runs best-first search with $f(n) = g(n) + 2\,h_2(n)$.

- ✏️ **Task 4a.** Before running: on puzzle C, will the solution be shorter than, the same as or longer than 26 moves? Will the search expand more or fewer nodes than A\* with $h_2$ (3,978)?

In [ ]:
best_first(EightPuzzle(C), lambda n: n.path_cost + 2 * h_manhattan(n.state));

- ✏️ **Task 4b.** Is $2 h_2$ admissible? Puzzle A gives a quick answer. Then run the check.

In [ ]:
check_admissible(lambda state: 2 * h_manhattan(state))

- ✏️ **Task 4c.** You got a solution 4 moves longer (15%) for about ten times less work. Name a situation where you would take that deal, and one where you would not.

## 5. Wrap-up (4 min)

UCS, greedy search and A\* differed only in $f$. A good heuristic cut the work on puzzle C from 169,741 nodes to 3,978 without losing the shortest solution. Overestimating cut it to 386, at the price of a longer path.

✏️ **Discussion** (in your group, then with the class)
1. Look at your rule of thumb from last week's lab (Task 1c). Can you write it as an $h(n)$? Is it admissible?
2. When you solved puzzle A by hand last week, did you search more like greedy search or more like A\*? Would that work on puzzle C?

**Next:** back to the slides. What you did in Part 4 has a name.